# 04 — Modeling and Hyperparameter Tuning

**Goal:** train at least two model families, use a proper time-based validation design, establish a baseline, and tune the models without leaking future information.

In [ ]:
from pathlib import Path
import sys
import warnings

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
# When notebooks are launched from notebooks/, move to repository root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW = PROJECT_ROOT / "data" / "raw" / "online_retail_II.xlsx"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES = PROJECT_ROOT / "figures"
MODELS = PROJECT_ROOT / "models"

DATA_PROCESSED.mkdir(exist_ok=True, parents=True)
FIGURES.mkdir(exist_ok=True, parents=True)
MODELS.mkdir(exist_ok=True, parents=True)

import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score
)

model_df = pd.read_csv(
    DATA_PROCESSED / "customer_modeling_data.csv",
    parse_dates=["PredictionDate"]
).sort_values("PredictionDate").reset_index(drop=True)

target = "return_30d"

feature_cols = [
    "RecencyDays", "TransactionCount", "PurchaseDays",
    "TotalSpend", "AverageOrderValue", "TotalQuantity",
    "UniqueProducts", "ActiveMonths", "AvgItemsPerTransaction",
    "PurchaseDaysPerMonth", "Country"
]

numeric_features = [c for c in feature_cols if c != "Country"]
categorical_features = ["Country"]

# Time split: 70% train, next 15% validation, last 15% test.
unique_dates = np.sort(model_df["PredictionDate"].unique())
train_end = unique_dates[int(len(unique_dates) * 0.70)]
val_end = unique_dates[int(len(unique_dates) * 0.85)]

train = model_df[model_df["PredictionDate"] < train_end].copy()
val = model_df[(model_df["PredictionDate"] >= train_end) & (model_df["PredictionDate"] < val_end)].copy()
test = model_df[model_df["PredictionDate"] >= val_end].copy()

print(train.shape, val.shape, test.shape)

## 1. Define preprocessing pipelines

In [ ]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features)
])

X_train, y_train = train[feature_cols], train[target]
X_val, y_val = val[feature_cols], val[target]
X_test, y_test = test[feature_cols], test[target]

## 2. Baseline

In [ ]:
# Trivial baseline: always predict the majority class from training.
majority_class = int(y_train.mode()[0])
baseline_pred = np.full(len(y_test), majority_class)

baseline = pd.Series({
    "accuracy": accuracy_score(y_test, baseline_pred),
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0)
})
display(baseline.to_frame("Baseline"))

## 3. Primary metric

**Primary metric:** F1-score.

F1 balances precision and recall and is more informative than accuracy when the positive repeat-purchase class is not the majority class.

## 4. Logistic Regression — baseline fit

In [ ]:
lr_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=42))
])

lr_pipe.fit(X_train, y_train)
lr_val_prob = lr_pipe.predict_proba(X_val)[:, 1]
lr_val_pred = (lr_val_prob >= 0.5).astype(int)

def metrics_row(y_true, y_pred, y_prob):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob),
        "pr_auc": average_precision_score(y_true, y_prob)
    }

lr_val_metrics = metrics_row(y_val, lr_val_pred, lr_val_prob)
display(pd.DataFrame(lr_val_metrics, index=["Logistic Regression"]))

## 5. Random Forest — baseline fit

In [ ]:
rf_pipe = Pipeline([
    ("prep", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipe.fit(X_train, y_train)
rf_val_prob = rf_pipe.predict_proba(X_val)[:, 1]
rf_val_pred = (rf_val_prob >= 0.5).astype(int)

rf_val_metrics = metrics_row(y_val, rf_val_pred, rf_val_prob)
display(pd.DataFrame(rf_val_metrics, index=["Random Forest"]))

## 6. Hyperparameter tuning

In [ ]:
# GridSearchCV is applied only to the training period.
# The validation period remains untouched for model comparison.
lr_grid = {
    "model__C": [0.01, 0.1, 1.0, 10.0],
    "model__solver": ["liblinear", "lbfgs"]
}

rf_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 8, 16],
    "model__min_samples_leaf": [1, 3, 8],
    "model__max_features": ["sqrt", 0.7]
}

# TimeSeriesSplit preserves temporal order within the training period.
tscv = TimeSeriesSplit(n_splits=4)

lr_search = GridSearchCV(
    lr_pipe, lr_grid, scoring="f1", cv=tscv, n_jobs=-1, return_train_score=True
)
lr_search.fit(X_train, y_train)

rf_search = GridSearchCV(
    rf_pipe, rf_grid, scoring="f1", cv=tscv, n_jobs=-1, return_train_score=True
)
rf_search.fit(X_train, y_train)

print("Best LR:", lr_search.best_params_, lr_search.best_score_)
print("Best RF:", rf_search.best_params_, rf_search.best_score_)

## 7. Save search results and best models

In [ ]:
pd.DataFrame(lr_search.cv_results_).to_csv(DATA_PROCESSED / "logistic_grid_results.csv", index=False)
pd.DataFrame(rf_search.cv_results_).to_csv(DATA_PROCESSED / "random_forest_grid_results.csv", index=False)

joblib.dump(lr_search.best_estimator_, MODELS / "logistic_regression_tuned.joblib")
joblib.dump(rf_search.best_estimator_, MODELS / "random_forest_tuned.joblib")

print("Tuned models and search results saved.")

### Important reporting requirement

In the report, show:
- the search space;
- the grid-search method;
- number of CV folds;
- the scoring metric;
- the trend across parameter settings;
- the selected configuration.

The final test set is not used for tuning.